# Experiment 80: TabNet 재학습 — R2 기반 GBDT + R2 PL

**목표**: LB 10.1201 (R2 단독) 돌파

**핵심 변경 (Exp 75 대비)**:
```
Exp 75:
  - GBDT: 72_top2 (LB 10.1287)
  - PL: 70b 기반
  - 결과: 75_g5 = LB 10.1284 (GBDT 대비 0.0003 개선)

Exp 80:
  - GBDT: R2 (LB 10.1201) ← 0.0086 더 좋은 base
  - PL: R2 예측 기반 합의 PL + confidence weight
  - 기대: R2 + TabNet → LB 10.118x
```

**실행 구조**:
```
Stage 1: TabNet 1-seed 빠른 탐색 (R2 상관관계 확인)  [~30분]
Stage 2: TabNet 5-seed 앙상블                         [~2시간]
Stage 3: R2 + TabNet 블렌딩 → 제출 파일 생성          [~5분]
```

In [ ]:
# Cell 1: 패키지 설치 및 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

!pip install -q lightgbm xgboost catboost pytorch-tabnet

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print("설치 완료")

In [ ]:
# Cell 2: Import 및 경로 설정
import pandas as pd
import numpy as np
import torch
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from sklearn.impute import SimpleImputer
from scipy.stats import pearsonr
import os, sys, warnings, gc

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
warnings.filterwarnings('ignore')

DATA_DIR    = '/content/drive/MyDrive/MyDrive/LightGBM'
TRAIN_PATH  = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH   = os.path.join(DATA_DIR, 'test.csv')
LAYOUT_PATH = os.path.join(DATA_DIR, 'layout_info.csv')

# R2 예측 (현재 PB, LB 10.1201) — PL + 블렌딩 base
R2_PATH = os.path.join(DATA_DIR, 'submission_79_r2.csv')

TARGET = 'avg_delay_minutes_next_30m'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

In [ ]:
# Cell 3: 유틸 함수
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if   c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:  df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
    return df

In [ ]:
# Cell 4: preprocess_data (Exp 67 원본 — 수정 금지)
def preprocess_data(train, test, layout):
    print("Preprocessing...")
    train = train.merge(layout, on='layout_id', how='left')
    test  = test.merge(layout,  on='layout_id', how='left')

    le = LabelEncoder()
    le.fit(pd.concat([train['layout_type'], test['layout_type']]).astype(str))
    train['layout_type'] = le.transform(train['layout_type'].astype(str))
    test['layout_type']  = le.transform(test['layout_type'].astype(str))

    for df in [train, test]:
        df['robot_utilization']    = df['robot_active']       / (df['robot_total']          + 1e-6)
        df['charger_utilization']  = df['robot_charging']     / (df['charger_count']         + 1e-6)
        df['aisle_pressure']       = df['congestion_score']   / (df['aisle_width_avg']        + 1e-6)
        df['intersection_density'] = df['intersection_count'] / (df['floor_area_sqm']         + 1e-6)
        df['pack_station_pressure']= df['order_inflow_15m']   / (df['pack_station_count']     + 1e-6)
        df['bottleneck_risk']      = df['congestion_score'] * df['intersection_density'] / (df['aisle_width_avg'] + 1e-6)
        df['task_intensity']       = df['order_inflow_15m']   / (df['robot_active']           + 1e-6)

    layout_num_cols = ['aisle_width_avg', 'intersection_count', 'robot_total']
    key_ops         = ['congestion_score', 'robot_active', 'order_inflow_15m']
    for l_col in layout_num_cols:
        for o_col in key_ops:
            train[f'{l_col}_x_{o_col}'] = train[l_col] * train[o_col]
            test[f'{l_col}_x_{o_col}']  = test[l_col]  * test[o_col]

    for col in ['congestion_score', 'low_battery_ratio', 'robot_active']:
        train[f'{col}_vel'] = train.groupby('scenario_id')[col].diff(1)
        test[f'{col}_vel']  = test.groupby('scenario_id')[col].diff(1)

    train['time_idx'] = train.groupby('scenario_id').cumcount()
    test['time_idx']  = test.groupby('scenario_id').cumcount()
    train = train.sort_values(['scenario_id', 'time_idx'])
    test  = test.sort_values(['scenario_id', 'time_idx'])

    SEQ_COLS = [
        'order_inflow_15m','unique_sku_15m','robot_active','robot_idle',
        'robot_charging','battery_mean','battery_std','low_battery_ratio',
        'charge_queue_length','avg_charge_wait','congestion_score',
        'max_zone_density','blocked_path_15m','near_collision_15m',
        'fault_count_15m','avg_recovery_time','task_reassign_15m',
        'replenishment_overlap','pack_utilization','loading_dock_util',
        'staging_area_util','label_print_queue'
    ]

    for col in SEQ_COLS:
        first_tr = train.groupby('scenario_id')[col].transform('first')
        train[f'{col}_vs_start']    = train[col] / (first_tr + 1e-6)
        train[f'{col}_delta_start'] = train[col] - first_tr
        first_ts = test.groupby('scenario_id')[col].transform('first')
        test[f'{col}_vs_start']    = test[col] / (first_ts + 1e-6)
        test[f'{col}_delta_start'] = test[col] - first_ts

    for col in SEQ_COLS:
        if col not in train.columns: continue
        for df in [train, test]:
            prev    = df.groupby('scenario_id')[col].shift(1)
            cum_max = prev.groupby(df['scenario_id']).cummax()
            cum_min = prev.groupby(df['scenario_id']).cummin()
            df[f'{col}_vs_cummax'] = df[col] / (cum_max + 1e-6)
            df[f'{col}_vs_cummin'] = df[col] / (cum_min.abs() + 1e-6)
            cum_range = cum_max - cum_min
            df[f'{col}_position_in_range'] = ((df[col] - cum_min) / (cum_range + 1e-3)).clip(0, 2)

    lag_cols = [
        'congestion_score','fault_count_15m','charge_queue_length',
        'low_battery_ratio','blocked_path_15m','avg_recovery_time',
        'near_collision_15m','pack_utilization'
    ]
    for col in lag_cols:
        if col not in train.columns: continue
        for df in [train, test]:
            for lag in [4,5,6,7]: df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [7,10,14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    SEQ_COLS_BASE = ['order_inflow_15m','unique_sku_15m','robot_active',
                     'low_battery_ratio','charge_queue_length','congestion_score','fault_count_15m']
    remaining = [c for c in SEQ_COLS_BASE if c not in lag_cols]
    for col in remaining:
        for df in [train, test]:
            for lag in [4,5]: df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [7,14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    for col in SEQ_COLS:
        for df in [train, test]:
            for lag in [8,10,12,15,20,24]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)

    remaining2 = [c for c in SEQ_COLS if c not in lag_cols]
    for col in remaining2:
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    tr_new = train['scenario_id'].values != np.roll(train['scenario_id'].values, 1); tr_new[0] = True
    ts_new = test['scenario_id'].values  != np.roll(test['scenario_id'].values,  1); ts_new[0]  = True
    for col in SEQ_COLS_BASE:
        for lag in [1,2]:
            tr_l = train[col].shift(lag).values.copy(); ts_l = test[col].shift(lag).values.copy()
            for l in range(lag):
                tr_l[np.roll(tr_new, l)] = np.nan; ts_l[np.roll(ts_new, l)] = np.nan
            train[f'{col}_lag{lag}'] = tr_l; test[f'{col}_lag{lag}'] = ts_l
        train[f'{col}_exp_mean'] = train.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())
        test[f'{col}_exp_mean']  = test.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())

    train['time_ratio'] = train.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    test['time_ratio']  = test.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    for df in [train, test]:
        df['congestion_ratio'] = df['congestion_score'] / (df['congestion_score_exp_mean'] + 1e-6)
        df['steps_remaining']  = df.groupby('scenario_id')['time_idx'].transform('max') - df['time_idx']

    train.fillna(0, inplace=True); test.fillna(0, inplace=True)
    return reduce_mem_usage(train), reduce_mem_usage(test)

In [ ]:
# Cell 5: apply_smoothed_te
def apply_smoothed_te(df_tr, df_val, target_col, k=30):
    global_mean = df_tr[target_col].mean()
    agg = df_tr.groupby('layout_id')[target_col].agg(['mean','std','median','count']).reset_index()
    agg['layout_mean'] = (agg['count'] * agg['mean'] + k * global_mean) / (agg['count'] + k)
    agg.rename(columns={'std':'layout_std','median':'layout_median','count':'layout_count'}, inplace=True)
    df_val = df_val.merge(agg[['layout_id','layout_mean','layout_std','layout_median','layout_count']], on='layout_id', how='left')
    df_tr  = df_tr.merge( agg[['layout_id','layout_mean','layout_std','layout_median','layout_count']], on='layout_id', how='left')
    df_val['layout_mean']   = df_val['layout_mean'].fillna(global_mean)
    df_val['layout_std']    = df_val['layout_std'].fillna(df_tr[target_col].std())
    df_val['layout_median'] = df_val['layout_median'].fillna(df_tr[target_col].median())
    df_val['layout_count']  = df_val['layout_count'].fillna(0)
    return df_tr, df_val, ['layout_mean','layout_std','layout_median','layout_count']

In [ ]:
# Cell 6: 공통 설정 + TabNet 파라미터
zero_imp_features = [
    'charge_queue_length','avg_charge_wait','charge_queue_length_lag2','fault_count_15m_lag2',
    'time_ratio','task_reassign_15m_vs_cummin','low_battery_ratio_vel','low_battery_ratio_lag2',
    'task_reassign_15m','blocked_path_15m_lag8','blocked_path_15m_lag10','near_collision_15m_lag8',
    'near_collision_15m_lag10','fault_count_15m_lag8','fault_count_15m_lag10','avg_recovery_time_lag8',
    'avg_recovery_time_lag10','task_reassign_15m_lag8','task_reassign_15m_lag10','replenishment_overlap_lag8',
    'replenishment_overlap_lag10','robot_charging_lag10','low_battery_ratio_lag8','low_battery_ratio_lag10',
    'charge_queue_length_lag8','charge_queue_length_lag10','avg_charge_wait_lag8','avg_charge_wait_lag10',
    'fault_count_15m_vs_cummax','fault_count_15m_vs_cummin','avg_recovery_time_vs_cummax','task_reassign_15m_vs_cummax',
    'low_battery_ratio_vs_cummax','low_battery_ratio_vs_cummin','charge_queue_length_vs_cummin','avg_charge_wait_vs_cummax',
    'blocked_path_15m_vs_cummax','near_collision_15m_vs_cummin','charge_queue_length_roll14_mean','task_reassign_15m_vs_start',
    'avg_recovery_time_lag5','avg_recovery_time_lag6','avg_recovery_time_lag7','near_collision_15m_lag4',
    'near_collision_15m_lag5','near_collision_15m_lag6','near_collision_15m_lag7','charge_queue_length_roll7_std',
    'charge_queue_length_roll10_mean','low_battery_ratio_lag4','low_battery_ratio_lag5','low_battery_ratio_lag7',
    'blocked_path_15m_lag4','blocked_path_15m_lag5','blocked_path_15m_lag6','blocked_path_15m_lag7',
    'label_print_queue_delta_start','robot_charging_lag15','low_battery_ratio_lag12','low_battery_ratio_lag15',
    'charge_queue_length_lag12','charge_queue_length_lag15','avg_charge_wait_lag12','avg_charge_wait_lag15',
    'congestion_score_lag12','max_zone_density_lag15','blocked_path_15m_lag12','fault_count_15m_lag4',
    'fault_count_15m_lag5','fault_count_15m_lag6','fault_count_15m_lag7','charge_queue_length_lag4',
    'charge_queue_length_lag5','charge_queue_length_lag6','charge_queue_length_lag7','charge_queue_length_roll7_mean',
    'blocked_path_15m_lag15','near_collision_15m_lag12','near_collision_15m_lag15','fault_count_15m_lag12',
    'fault_count_15m_lag15','avg_recovery_time_lag12','avg_recovery_time_lag15','task_reassign_15m_lag12',
    'task_reassign_15m_lag15','replenishment_overlap_lag12','replenishment_overlap_lag15','charge_queue_length_position_in_range',
    'avg_charge_wait_position_in_range','congestion_score_position_in_range','blocked_path_15m_position_in_range',
    'near_collision_15m_position_in_range','fault_count_15m_position_in_range','avg_recovery_time_position_in_range',
    'task_reassign_15m_position_in_range','replenishment_overlap_position_in_range','label_print_queue_position_in_range',
    'replenishment_overlap_vs_cummin','staging_area_util_vs_cummax','battery_mean_position_in_range',
    'low_battery_ratio_position_in_range','label_print_queue_lag15','charge_queue_length_roll20_std',
    'charge_queue_length_vs_start','charge_queue_length_delta_start','avg_charge_wait_vs_start','avg_charge_wait_delta_start'
]

gkf   = GroupKFold(n_splits=5)
seeds = [42, 123, 2026, 777, 1004]  # TabNet은 5-seed (Exp 75와 동일)

def target_forward(y): return np.log1p(y)
def target_inverse(p): return np.expm1(p)

# ── TabNet 하이퍼파라미터 (Exp 75 확정값) ──
TABNET_PARAMS = {
    'n_d': 64, 'n_a': 64, 'n_steps': 5,
    'gamma': 1.3, 'lambda_sparse': 1e-4,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': {'lr': 2e-3, 'weight_decay': 1e-5},
    'scheduler_fn': torch.optim.lr_scheduler.CosineAnnealingLR,
    'scheduler_params': {'T_max': 50, 'eta_min': 1e-5},
    'mask_type': 'entmax',
    'device_name': DEVICE,
    'verbose': 0,
}
TABNET_FIT = {
    'max_epochs': 200, 'patience': 30,
    'batch_size': 4096, 'virtual_batch_size': 512,
    'num_workers': 0, 'drop_last': False,
    'eval_metric': ['mae'],
}

print("설정 완료")

In [ ]:
# Cell 7: TabNet 전처리 함수 (Exp 75 원본)
def tabnet_preprocess(X_tr_raw, X_val_raw, X_te_raw, feats, seed=42):
    imputer = SimpleImputer(strategy='median')
    X_tr_imp  = imputer.fit_transform(X_tr_raw[feats].values.astype(np.float32))
    X_val_imp = imputer.transform(X_val_raw[feats].values.astype(np.float32))
    X_te_imp  = imputer.transform(X_te_raw[feats].values.astype(np.float32))

    qt = QuantileTransformer(
        n_quantiles=1000, output_distribution='uniform', random_state=seed)
    X_tr_s  = qt.fit_transform(X_tr_imp).astype(np.float32)
    X_val_s = qt.transform(X_val_imp).astype(np.float32)
    X_te_s  = qt.transform(X_te_imp).astype(np.float32)
    return X_tr_s, X_val_s, X_te_s

print("TabNet 전처리 함수 정의 완료")

In [ ]:
# Cell 8: 데이터 로드 + 전처리 + R2 기반 pseudo 구성
print("--- Experiment 80: TabNet + R2 ---")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
layout    = pd.read_csv(LAYOUT_PATH)

train_layouts  = set(train_raw['layout_id'].unique())
test_layouts   = set(test_raw['layout_id'].unique())
seen_layouts   = train_layouts & test_layouts

train, test = preprocess_data(train_raw, test_raw, layout)

EXCLUDE_COLS    = ['ID','layout_id','scenario_id',TARGET,'is_seen','is_pseudo','pseudo_label']
features_base   = [c for c in train.columns if c not in EXCLUDE_COLS]
features_pruned = [f for f in features_base if f not in zero_imp_features]

train['is_seen'] = train['layout_id'].isin(seen_layouts)
test['is_seen']  = test['layout_id'].isin(seen_layouts)

# ── R2 예측을 PL로 사용 ──
pred_r2 = pd.read_csv(R2_PATH)
test = test.merge(pred_r2.rename(columns={TARGET: 'pseudo_label'}), on='ID', how='left')

# 전략 C pseudo 구성 (R2 기반)
train_std      = train[TARGET].std()
train_q01      = train[TARGET].quantile(0.01)
train_q99      = train[TARGET].quantile(0.99)
unseen_test    = test[~test['is_seen']]
layout_std_map = unseen_test.groupby('layout_id')['pseudo_label'].std()
low_var_lay    = layout_std_map[layout_std_map < train_std * 0.3].index.tolist()
mask_low_var   = test['layout_id'].isin(low_var_lay) & ~test['is_seen']
mask_extreme   = (test['pseudo_label'] < train_q01) | (test['pseudo_label'] > train_q99)

pseudo_mask = (test['is_seen'] & ~mask_extreme) | (~test['is_seen'] & ~mask_low_var & ~mask_extreme)
pseudo_set  = test[pseudo_mask].copy()
pseudo_set[TARGET]      = pseudo_set['pseudo_label']
pseudo_set['is_pseudo'] = True
shared_cols  = [c for c in train.columns if c in pseudo_set.columns]
train['is_pseudo'] = False
if 'is_pseudo' not in shared_cols:
    shared_cols.append('is_pseudo')
if TARGET not in shared_cols:
    shared_cols.append(TARGET)
train_combined = pd.concat([train[shared_cols], pseudo_set[shared_cols]], ignore_index=True)
train_combined['is_pseudo'] = train_combined['is_pseudo'].fillna(False)

# R2 test 예측 로드 (블렌딩용)
pred_r2_test = pred_r2.set_index('ID').loc[test['ID'].values][TARGET].values

orig_mask = ~train_combined['is_pseudo'].fillna(False).values
y_orig    = train_combined[TARGET].values[orig_mask]

print(f"피처: {len(features_pruned)}개")
print(f"원본 train: {len(train)}행")
print(f"통합 train: {len(train_combined)}행 (pseudo {len(train_combined)-len(train)}행)")
print(f"R2 PL mean: {test['pseudo_label'].mean():.4f}")

In [ ]:
# Cell 9: [Stage 1] TabNet 1-seed 빠른 탐색
#
# 목적: R2와의 상관관계 + TabNet 단독 OOF 확인
# 소요 시간: ~30분

print("[Stage 1] TabNet 단독 OOF 측정 (seed=42)")
print("="*60)

oof_tab_s1  = np.zeros(len(train_combined))
test_tab_s1 = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(train_combined, train_combined[TARGET],
                  groups=train_combined['layout_id'])):

    X_tr_raw  = train_combined.iloc[tr_idx].copy()
    y_tr_raw  = train_combined.iloc[tr_idx][TARGET].values
    X_val_raw = train_combined.iloc[val_idx].copy()
    y_val_raw = train_combined.iloc[val_idx][TARGET].values

    # CV-safe TE
    X_tr_raw, X_val_raw, te_cols = apply_smoothed_te(X_tr_raw, X_val_raw, TARGET, k=30)
    X_tr_raw.fillna(0, inplace=True); X_val_raw.fillna(0, inplace=True)
    _, X_te_raw, _ = apply_smoothed_te(X_tr_raw, test.copy(), TARGET, k=30)
    X_te_raw.fillna(0, inplace=True)
    feats = features_pruned + te_cols

    y_tr_t  = target_forward(y_tr_raw)
    y_val_t = target_forward(y_val_raw)

    # TabNet 전처리
    X_tr_s, X_val_s, X_te_s = tabnet_preprocess(
        X_tr_raw, X_val_raw, X_te_raw, feats, seed=42)

    # TabNet 학습
    np.random.seed(42); torch.manual_seed(42)
    model = TabNetRegressor(**{**TABNET_PARAMS, 'seed': 42})
    model.fit(
        X_tr_s, y_tr_t.reshape(-1, 1),
        eval_set=[(X_val_s, y_val_t.reshape(-1, 1))],
        **TABNET_FIT
    )

    pred_val  = np.maximum(target_inverse(model.predict(X_val_s).flatten()), 0)
    pred_test = np.maximum(target_inverse(model.predict(X_te_s).flatten()), 0)
    oof_tab_s1[val_idx] = pred_val
    test_tab_s1 += pred_test / 5

    fold_mae = mean_absolute_error(y_val_raw, pred_val)
    print(f"  Fold {fold+1} MAE={fold_mae:.4f} (best_ep={model.best_epoch})")
    del model; gc.collect(); torch.cuda.empty_cache()

tab_s1_mae = mean_absolute_error(y_orig, oof_tab_s1[orig_mask])
corr_r2_tab, _ = pearsonr(pred_r2_test, test_tab_s1)

print(f"\n[Stage 1 결과]")
print(f"  TabNet OOF MAE: {tab_s1_mae:.4f}")
print(f"  R2 ↔ TabNet 상관관계 (test): {corr_r2_tab:.4f}")
print(f"  참고 — Exp 75 TabNet OOF: ~9.2, GBDT↔TabNet: ~0.98")

if corr_r2_tab < 0.96:
    print(f"  ✅ 상관관계 {corr_r2_tab:.3f} < 0.96 → 강한 블렌딩 효과 기대!")
elif corr_r2_tab < 0.99:
    print(f"  🟡 상관관계 {corr_r2_tab:.3f} → γ=0.03~0.10 범위 탐색")
else:
    print(f"  🔴 상관관계 {corr_r2_tab:.3f} ≥ 0.99 → 블렌딩 효과 제한적")

In [ ]:
# Cell 10: [Stage 2] TabNet 5-seed 앙상블
#
# Stage 1에서 합리적인 상관관계 확인 후 실행
# 소요 시간: ~2시간

print("[Stage 2] TabNet 5-Seed 앙상블")
print(f"  seeds: {seeds}")
print(f"  pseudo: R2 기반, weight Seen=1.0, Unseen=0.2\n")

def make_tab_weight(df, w_orig=1.0, w_seen=1.0, w_unseen=0.2):
    is_pseudo = df['is_pseudo'].fillna(False).values
    is_seen   = df['is_seen'].values
    return np.where(~is_pseudo, w_orig,
           np.where(is_seen, w_seen, w_unseen))

oof_tab   = np.zeros(orig_mask.sum())
test_tab  = np.zeros(len(test))

for seed_idx, seed in enumerate(seeds):
    print(f"\n{'='*25} TabNet Seed {seed} ({seed_idx+1}/{len(seeds)}) {'='*25}")

    oof_seed  = np.zeros(len(train_combined))
    test_seed = np.zeros(len(test))

    for fold, (tr_idx, val_idx) in enumerate(
            gkf.split(train_combined, train_combined[TARGET],
                      groups=train_combined['layout_id'])):

        X_tr_raw  = train_combined.iloc[tr_idx].copy()
        y_tr_raw  = train_combined.iloc[tr_idx][TARGET].values
        sw_tr     = make_tab_weight(train_combined.iloc[tr_idx])
        X_val_raw = train_combined.iloc[val_idx].copy()
        y_val_raw = train_combined.iloc[val_idx][TARGET].values

        # CV-safe TE
        X_tr_raw, X_val_raw, te_cols = apply_smoothed_te(X_tr_raw, X_val_raw, TARGET, k=30)
        X_tr_raw.fillna(0, inplace=True); X_val_raw.fillna(0, inplace=True)
        _, X_te_raw, _ = apply_smoothed_te(X_tr_raw, test.copy(), TARGET, k=30)
        X_te_raw.fillna(0, inplace=True)
        feats = features_pruned + te_cols

        y_tr_t  = target_forward(y_tr_raw)
        y_val_t = target_forward(y_val_raw)

        X_tr_s, X_val_s, X_te_s = tabnet_preprocess(
            X_tr_raw, X_val_raw, X_te_raw, feats, seed=seed)

        np.random.seed(seed); torch.manual_seed(seed)
        model = TabNetRegressor(**{**TABNET_PARAMS, 'seed': seed})
        model.fit(
            X_tr_s, y_tr_t.reshape(-1, 1),
            eval_set=[(X_val_s, y_val_t.reshape(-1, 1))],
            weights=sw_tr,
            **TABNET_FIT
        )

        pred_val  = np.maximum(target_inverse(model.predict(X_val_s).flatten()), 0)
        pred_test = np.maximum(target_inverse(model.predict(X_te_s).flatten()), 0)

        oof_seed[val_idx] = pred_val
        test_seed        += pred_test / 5

        fold_mae = mean_absolute_error(y_val_raw, pred_val)
        print(f"  F{fold+1} MAE={fold_mae:.4f} ep={model.best_epoch}",
              end=' | ', flush=True)
        del model; gc.collect(); torch.cuda.empty_cache()

    orig_indices = np.where(orig_mask)[0]
    seed_mae = mean_absolute_error(y_orig, oof_seed[orig_mask])
    oof_tab  += oof_seed[orig_indices] / len(seeds)
    test_tab += test_seed / len(seeds)
    print(f"\n  Seed {seed} OOF: {seed_mae:.4f}")
    gc.collect()

tab_final_mae = mean_absolute_error(y_orig, oof_tab)
print(f"\n[Stage 2 완료] TabNet 5-seed OOF MAE: {tab_final_mae:.4f}")

In [ ]:
# Cell 11: [Stage 3] R2 + TabNet 상관관계 + 블렌딩 + 제출

print("[Stage 3] R2 + TabNet 블렌딩")
print("="*60)

corr_final, _ = pearsonr(pred_r2_test, test_tab)
print(f"  R2 ↔ TabNet(5-seed) 상관관계: {corr_final:.4f}")

# ── 블렌딩 γ 탐색 ──
if corr_final < 0.92:
    gammas = [0.10, 0.15, 0.20, 0.25, 0.30]
elif corr_final < 0.96:
    gammas = [0.05, 0.08, 0.10, 0.15, 0.20]
else:
    gammas = [0.03, 0.05, 0.08, 0.10]

print(f"  γ 탐색 범위: {gammas}")
print()

saved_files = []

# TabNet 단독 저장
fname = 'submission_80_tabnet_only.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: np.maximum(test_tab, 0)}).to_csv(fpath, index=False)
saved_files.append(('TabNet 단독', fname))
print(f"  ✅ {fname}")

# R2 + TabNet 블렌딩
for gamma in gammas:
    blend = (1 - gamma) * pred_r2_test + gamma * test_tab
    blend = np.maximum(blend, 0)
    fname = f'submission_80_g{int(gamma*100)}.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    diff = np.abs(blend - pred_r2_test).mean()
    saved_files.append((f'R2+TabNet γ={gamma}', fname))
    print(f"  ✅ {fname} — {int((1-gamma)*100)}%R2 + {int(gamma*100)}%TabNet (diff={diff:.4f})")

print(f"\n총 {len(saved_files)}개 파일 저장")

In [ ]:
# Cell 12: 최종 결과 요약

print("\n" + "="*70)
print("  [Exp 80: TabNet + R2 — 최종 결과]")
print("="*70)

print(f"\n  {'모델':<35} {'OOF MAE':>10}")
print("-"*50)
print(f"  {'R2 GBDT (Exp 79)':>35} {'8.6906':>10}")
print(f"  {'TabNet 5-seed (Exp 80)':>35} {tab_final_mae:>10.4f}")
print("-"*50)
print(f"  R2 ↔ TabNet 상관관계: {corr_final:.4f}")
print(f"  참고 — Exp 75 (72_top2 ↔ TabNet): ~0.98")

print(f"\n  현재 PB: LB 10.1201 (R2 단독)")

print(f"\n  {'─'*55}")
print(f"  📋 제출 순서:")
print(f"  {'─'*55}")

if corr_final < 0.96:
    rec_g = gammas[2]
    print(f"  1순위: submission_80_g{int(rec_g*100)}.csv")
    print(f"         (상관관계 {corr_final:.3f} 낮음 → γ={rec_g} 적극적)")
elif corr_final < 0.99:
    rec_g = gammas[1]
    print(f"  1순위: submission_80_g{int(rec_g*100)}.csv")
    print(f"         (상관관계 {corr_final:.3f} → γ={rec_g})")
else:
    rec_g = gammas[0]
    print(f"  1순위: submission_80_g{int(rec_g*100)}.csv")
    print(f"         (상관관계 {corr_final:.3f} 높음 → γ={rec_g} 보수적)")

print(f"  2순위: submission_80_g{int(gammas[0]*100)}.csv")
print(f"         (γ={gammas[0]} 보수적 블렌딩)")
print()
print(f"  🔍 판단 기준:")
print(f"  ┌───────────────────────────────────────────────────────┐")
print(f"  │ LB < 10.1201 → TabNet 블렌딩 효과 확인!             │")
print(f"  │   → γ 범위 세밀 탐색                                │")
print(f"  │                                                      │")
print(f"  │ LB ≈ 10.1201 → R2가 이미 최적, TabNet 추가 효과 없음│")
print(f"  │   → 다른 전략(post-processing 등) 탐색              │")
print(f"  └───────────────────────────────────────────────────────┘")
print("="*70)